# Read and Write Data

So far, we have created small DataFrames in the notebook. In practice, Spark reads data from storage and writes useful results for other people and processes to use.

---
## Learning objectives

By the end of this notebook, you will be able to:

- read a CSV from Lakehouse Files using an explicit DDL schema;
- inspect the data and schema after reading;
- write a DataFrame as a managed Delta table; and
- read a managed table back into a DataFrame.

---
## Before running this notebook

Attach a Lakehouse to this notebook. Upload the supplied `orders.csv`, `customers.csv`, and `products.csv` files into its `Files/pyspark_training` folder. The first cell below stores that folder path in one place.

In [9]:
spark.stop()

In [10]:
import os
import sys

os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["PYSPARK_PYTHON"] = sys.executable

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("local-test")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.driver.host", "127.0.0.1")
    .getOrCreate()
)

In [16]:
input_path = 'C:\\Users\\justin.diener\\OneDrive\\LACO\\Given Training\\Pyspark\\Wills pyspark git code\\wills_pyspark_training\\pyspark_training_interns\\retail_data\\'
orders_path = f"{input_path}orders.csv" 
customers_path = f"{input_path}customers.csv"

---
## Read a CSV with a schema

The schema tells Spark how to interpret each CSV column. Providing it explicitly avoids relying on schema inference and makes the expected data contract clear.

In [31]:
orders_schema = """
    order_id INT,
    customer_id STRING,
    product_id STRING,
    quantity INT,
    unit_price DOUBLE,
    order_date DATE
"""

orders = (
    spark.read
    .option("header", True)
    .schema(orders_schema)
    .csv(orders_path)
)

orders.show()
orders.printSchema()

+--------+-----------+----------+--------+----------+----------+
|order_id|customer_id|product_id|quantity|unit_price|order_date|
+--------+-----------+----------+--------+----------+----------+
|    1001|       C001|      P001|       2|      18.5|2026-01-05|
|    1002|       C002|      P002|       1|     750.0|2026-01-06|
|    1003|       C001|      P003|       3|      12.0|2026-01-08|
|    1004|       C003|      P001|       5|      18.5|2026-01-10|
|    1005|       C004|      P004|       2|      85.0|2026-01-12|
|    1006|       C002|      P003|       4|      12.0|2026-01-14|
|    1007|       C003|      P002|       1|     750.0|2026-01-16|
|    1008|       C004|      P001|       3|      18.5|2026-01-18|
+--------+-----------+----------+--------+----------+----------+

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- 

---
## Write a managed Delta table

Delta is the Lakehouse table format used by Fabric.

### Delta files vs. a managed table

`.save("Files/orders_delta")` writes Delta files to a path only. In contrast, `.saveAsTable("retail_orders")` writes the Delta data and registers the named table in the Lakehouse catalog, so it can later be read with `spark.table("retail_orders")` or SQL with no path supplied, this creates a managed table. 

`mode("overwrite")` lets us safely rerun this training notebook by replacing only the course table named `retail_orders`.

In [ ]:
# Will only update the data in the retail_orders delta file
(
    orders.write
    .mode("overwrite")
    .format("delta")
    .save(f"{orders_path}delta_retail_orders")
)

In [ ]:
# will create a delta table called retail_orders in the default database
(
    orders.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("retail_orders")
)

---
## Read the table back

Once saved, a managed table can be read with `spark.table()`. This is how downstream notebooks can use a reliable named dataset.

In [ ]:
retail_orders = spark.table("retail_orders")
retail_orders.show()

---
## Your turn

Read `customers.csv` using `customers_path` and an explicit DDL schema. Name the DataFrame `customers`, preview it, then write it as a managed Delta table named `retail_customers`.

In [ ]:
# Write your solution here.

---
## Preparing for the afternoon exercise

The same three CSVs will be used in the capstone. You will read them, transform and join the data, calculate a summary, and write the final result as a Delta table.